In [16]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [24]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            SELECT Identificacion,Periodo, cod_unidad, Estado_Alumno as Estado_simulado 
            FROM CUN_REPOSITORIO.DESERCION.Python_CUN_Simulado_cambios_ciclo
                

        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_a = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [3]:
df_a.columns

Index(['Identificacion', 'Periodo', 'ciclo_sugerido', 'Cod_unidad_sug',
       'pensum_limpio', 'Estado_simulado'],
      dtype='object')

In [19]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            SELECT  * 
            FROM OPENQUERY([172.16.1.175], 
                '
				SELECT DISTINCT
                        a.COD_PERIODO,
                        a.TIP_NOTA,
                        a.OBS_NOTA,
                        b.NUM_IDENTIFICACION 
				FROM SINU.SRC_HIS_ACADEMICA a
				LEFT JOIN SINU.SRC_ALUM_PROGRAMA c ON a.ID_ALUM_PROGRAMA = c.ID_ALUM_PROGRAMA
                LEFT JOIN SINU.BAS_TERCERO b ON b.ID_TERCERO = c.ID_TERCERO
                WHERE a.TIP_NOTA IN (''H'',''HE'',''N'') AND a.COD_PERIODO NOT LIKE ''%I%'' 
                GROUP BY a.COD_PERIODO,
                        a.TIP_NOTA,
                        a.OBS_NOTA,
                        b.NUM_IDENTIFICACION
						 ')    

        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_b = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [6]:
df_b.columns

Index(['COD_PERIODO', 'TIP_NOTA', 'OBS_NOTA', 'NUM_IDENTIFICACION',
       'MATERIAS_APROBADAS'],
      dtype='object')

In [20]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            select NUM_IDENTIFICACION, CICLO, COD_UNIDAD,
            CASE WHEN COD_PENSUM LIKE '%D' THEN REPLACE (COD_PENSUM ,'D','')
                WHEN COD_PENSUM LIKE '%N' THEN REPLACE (COD_PENSUM ,'N','')
                ELSE COD_PENSUM END AS PENSUM_L 
      
        from openquery([172.16.1.175],
        'select * from iceberg.CICLOS_X_ESTUDIANTE')
        Where cant_mat_homologadas is not null        

        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_c = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [7]:
df_c.columns

Index(['NUM_IDENTIFICACION', 'CANT_MATERIAS', 'CANT_MAT_APRO',
       'CANT_MAT_HOMOLOGADAS', 'CICLO', 'COD_UNIDAD', 'PENSUM_L'],
      dtype='object')

In [8]:
df_b = df_b.rename(columns={'NUM_IDENTIFICACION':'Identificacion'})
df_c = df_c.rename(columns={'NUM_IDENTIFICACION':'Identificacion'})

In [9]:
df_unido = pd.concat([df_a, df_b, df_c], ignore_index=True)

In [ ]:
display(df_unido)

In [16]:
df_unido.to_excel("UNOo.xlsx")

In [13]:
df_unidos = pd.merge(df_a, df_b, how="left", left_on='Identificacion' ,right_on='Identificacion' )

In [14]:
df_unidoss = pd.merge(df_unidos, df_b, how="left", left_on='Identificacion' ,right_on='Identificacion' )

In [8]:
df_c.shape

(12, 7)

In [9]:
df_a.shape

(8, 6)

In [10]:
df_b.shape

(17, 5)

In [21]:
display(df_c)

,NUM_IDENTIFICACION,CICLO,COD_UNIDAD,PENSUM_L
0,1000590507,TECNOLOGO,TCH12,1035
1,1000590507,TECNICO,TCH12,1035
2,1000590518,TECNICO,1TS01,1036
3,1000590534,TECNICO,1TC07,8014
4,1000590537,TECNICO,VTC07,1120
...,...,...,...,...
986957,83183336,TECNICO,1PV02,1011
986958,83183336,TECNICO,1TV01,1010
986959,83183336,TECNOLOGO,1TV02,1011
986960,83183336,PROFESIONAL,1TV02,1011


In [22]:
display(df_b)

,COD_PERIODO,TIP_NOTA,OBS_NOTA,NUM_IDENTIFICACION
0,2008A,H,"migracion uxxi -> sinu, fecha: 01-JUN-11",8110712
1,2008B,N,"migracion uxxi -> sinu, fecha: 01-JUN-11",22467575
2,2010A,N,"migracion uxxi -> sinu, fecha: 28-MAY-11",24674790
3,2010B,N,ACTUALIZACION HISTORIA 2013B,39490896
4,2011A,N,"migracion uxxi -> sinu, fecha: 01-JUN-11",7601997
...,...,...,...,...
2995849,25V02,HE,HOMOLOGACION EXTERNA#2806865 SXRIVERA,1000268543
2995850,2025B,HE,HOMOLOGACION EXTERNA #2785617 SGRANADOS,1001062999
2995851,2015A,H,Homologación,80193073
2995852,25V02,HE,HOMOLOGACIÓN EXTERNA #2810986 KOPARRADO,1075259025


In [25]:
display(df_a)

,Identificacion,Periodo,cod_unidad,Estado_simulado
0,00122200027,2018A,1TC02,Traslado
1,0401568233,2019A,TIP12,Graduado
2,0401568233,2019B,TIP12,Graduado
3,0401568233,2022A,TIP12,Graduado
4,0401568233,2022B,TIP12,Graduado
...,...,...,...,...
864538,9976845,2025A,1TC71,Activo
864539,9976918,2013A,TAM06,Graduado
864540,9976918,2017A,TAM06,Graduado
864541,9977702,21V06,TVM05,Activo


In [26]:
df_unido_1 = pd.merge(df_c, df_a, how="inner", left_on=['NUM_IDENTIFICACION','COD_UNIDAD'],right_on=['Identificacion','cod_unidad'])

In [33]:
df_unido_1

,NUM_IDENTIFICACION,CICLO,COD_UNIDAD,PENSUM_L,Periodo,Estado_simulado
0,1000590507,TECNOLOGO,TCH12,1035,2021B,Activo
1,1000590507,TECNOLOGO,TCH12,1035,2022A,Activo
2,1000590507,TECNOLOGO,TCH12,1035,2019B,Activo
3,1000590507,TECNOLOGO,TCH12,1035,2020A,Activo
4,1000590507,TECNOLOGO,TCH12,1035,2021A,Activo
...,...,...,...,...,...,...
2252210,83183190,TECNICO,TNV06,1071,2014A,Graduado
2252211,83183190,TECNICO,TNV06,1071,2016A,Graduado
2252212,83183190,TECNICO,TNV06,1071,2016B,Graduado
2252213,83183190,TECNICO,TNV06,1071,2015B,Graduado


In [32]:
df_unido_1.drop(columns=["Identificacion","cod_unidad"],inplace=True)

In [34]:
df_unido_2 = pd.merge(df_unido_1, df_b, how="inner", left_on=['NUM_IDENTIFICACION','Periodo'],right_on=['NUM_IDENTIFICACION','COD_PERIODO'])

In [73]:
df_unido_3.drop(columns=["Categoria"],inplace=True)

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_23128\1895627460.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unido_3.drop(columns=["Categoria"],inplace=True)


In [41]:
df_unido_3 = df_unido_2[df_unido_2["NUM_IDENTIFICACION"] == "1000590507"]

In [74]:
df_unido_3

,NUM_IDENTIFICACION,CICLO,COD_UNIDAD,PENSUM_L,Periodo,Estado_simulado,TIP_NOTA,OBS_NOTA
0,1000590507,TECNOLOGO,TCH12,1035,2021B,Activo,N,Traslado de notas
1,1000590507,TECNOLOGO,TCH12,1035,2022A,Activo,N,Traslado de notas
2,1000590507,TECNOLOGO,TCH12,1035,2019B,Activo,N,Traslado de notas
3,1000590507,TECNOLOGO,TCH12,1035,2020A,Activo,N,Traslado de notas
4,1000590507,TECNOLOGO,TCH12,1035,2021A,Activo,N,Traslado de notas
5,1000590507,TECNICO,TCH12,1035,2021B,Activo,N,Traslado de notas
6,1000590507,TECNICO,TCH12,1035,2022A,Activo,N,Traslado de notas
7,1000590507,TECNICO,TCH12,1035,2019B,Activo,N,Traslado de notas
8,1000590507,TECNICO,TCH12,1035,2020A,Activo,N,Traslado de notas
9,1000590507,TECNICO,TCH12,1035,2021A,Activo,N,Traslado de notas


In [88]:
agrupado = df_unido_3.groupby("NUM_IDENTIFICACION")["CICLO"].agg(set).reset_index()

def categorizar(ciclos_unicos):

    if {'PROFESIONAL'}.issubset(ciclos_unicos):
        return "Reconocimiento de titulo"
    
    elif {'TECNICO','TECNOLOGO'}.issubset(ciclos_unicos):
        return "Reconocimiento de tecnico"
    else:
        return "Reconocimiento de tecnico"
    
agrupado['Categoria'] = agrupado["CICLO"].apply(categorizar)
    

In [89]:
df_final2 = df_unido_3.merge(agrupado[['NUM_IDENTIFICACION', 'Categoria']], on='NUM_IDENTIFICACION', how='left')

In [90]:
df_final2.drop(columns={"Periodo"}, inplace=True)

In [91]:
display(df_final2)

,NUM_IDENTIFICACION,CICLO,COD_UNIDAD,PENSUM_L,Estado_simulado,TIP_NOTA,OBS_NOTA,Categoria
0,1000590507,TECNOLOGO,TCH12,1035,Activo,N,Traslado de notas,Reconocimiento de tecnico
1,1000590507,TECNOLOGO,TCH12,1035,Activo,N,Traslado de notas,Reconocimiento de tecnico
2,1000590507,TECNOLOGO,TCH12,1035,Activo,N,Traslado de notas,Reconocimiento de tecnico
3,1000590507,TECNOLOGO,TCH12,1035,Activo,N,Traslado de notas,Reconocimiento de tecnico
4,1000590507,TECNOLOGO,TCH12,1035,Activo,N,Traslado de notas,Reconocimiento de tecnico
5,1000590507,TECNICO,TCH12,1035,Activo,N,Traslado de notas,Reconocimiento de tecnico
6,1000590507,TECNICO,TCH12,1035,Activo,N,Traslado de notas,Reconocimiento de tecnico
7,1000590507,TECNICO,TCH12,1035,Activo,N,Traslado de notas,Reconocimiento de tecnico
8,1000590507,TECNICO,TCH12,1035,Activo,N,Traslado de notas,Reconocimiento de tecnico
9,1000590507,TECNICO,TCH12,1035,Activo,N,Traslado de notas,Reconocimiento de tecnico
